# Student fine-tuning: Qwen3-ASR Khmer LoRA
Two completed student LoRA experiments use the same 124 FLEURS validation clips. The original run trained on 531 clips; the expanded run trained on 883 official train clips of at most 15 seconds. The script never loads held-out test audio.

**Current decision: do not add more epochs to this recipe.** Expanded epoch 1: CER 14.52%, ICU Khmer WER 32.26%. Epoch 2: CER 15.08%, WER 34.02%. Falling training loss with worsening validation is consistent with overfitting. Neither epoch beat the recorded public baseline (14.29% / 32.07%). Both error rates must be below 20% to meet the reported target; WER remains above it.

This notebook is retained for reproduction and a future justified experiment. Training is disabled by default. See `results/diagnostic/qwen_lora_expanded_v2.json` for the complete validation evidence. The expanded run is separate from the original matched-training-size comparison.


In [ ]:
!nvidia-smi
!pip install -q qwen-asr datasets torchcodec pyicu-wheels==2.15.2 jiwer soundfile accelerate peft
!pip uninstall -y torchao
from pathlib import Path
import subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training.')
repo = Path('/content/khmer_asr')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Seypa-47/khmer_asr.git', str(repo)], check=True)
official = Path('/content/Qwen3-ASR')
if not official.exists():
    subprocess.run(['git', 'clone', 'https://github.com/QwenLM/Qwen3-ASR.git', str(official)], check=True)
subprocess.run(['git', '-C', str(official), 'checkout', '7c6daf77a2421100f5fb066495372c00129d39ff'], check=True)
%cd /content/khmer_asr


Mount Drive only after reviewing the code above. If it fails, the code uses runtime storage and prints a warning. Runtime storage disappears when Colab disconnects; download the adapter folder or reproduce the run locally before disconnecting.


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
    output_root = Path('/content/drive/MyDrive/khmer_asr_final_runs')
except ValueError:
    output_root = Path('/content/khmer_asr_runtime_checkpoints')
    print('Drive mount failed. Download adapters before the runtime ends or reproduce locally.')
output_root.mkdir(parents=True, exist_ok=True)


In [ ]:
# Enable only for a deliberate new run. Do not overwrite completed evidence.
RUN_TRAINING = False
EXPERIMENT = 'expanded_v2'  # 'original_v1' or 'expanded_v2'
configs = {
    'original_v1': ('results/matched_fleurs_split.json', '1', '1e-4'),
    'expanded_v2': ('results/diagnostic/qwen_expanded_train_split.json', '2', '5e-5'),
}
manifest, epochs, lr = configs[EXPERIMENT]
output_dir = output_root / (EXPERIMENT + '_reproduction')
if RUN_TRAINING:
    if output_dir.exists():
        raise FileExistsError(f'Use a new output directory to preserve existing checkpoints: {output_dir}')
    subprocess.run([
        'python', '-u', 'src/train_qwen_lora.py', '--manifest', manifest,
        '--output-dir', str(output_dir), '--epochs', epochs,
        '--lr', lr, '--grad-acc', '4',
    ], check=True)
else:
    print('Training disabled: completed experiments did not meet the WER target. Review their results before choosing a new experiment.')


Compare validation scores with the recorded publisher checkpoint on the same 124 clips; differences across GPU types can affect predictions. Select a model using validation before evaluating the held-out test once. Keep public baseline, student training, validation, and test claims separate.

Historical runs used `src/train_qwen_lora.py` at commit `a2fb1b6`; the current script corrects the last partial gradient accumulation group's divisor. Re-running current code is therefore a revised reproduction, not a promise to recover identical historical numbers. Epoch folders save adapters, not the optimizer/scaler/RNG state needed for exact resumption. Original adapters remain in the local `models/` directory, outside Git. Google Drive upload success must be verified before claiming a cloud backup exists.
